In [4]:
import pandas as pd
print('-'*100)
df1 = pd.read_feather("../output/URE(100dim)_output/Adult Products_URE.feather")
print(df1[df1['UserID'] == 2815])
print(len(df1))
value_counts = df1["UserID"].value_counts()
print("고유값",len(value_counts))

print('-'*100)
df1 = pd.read_feather("../output/UI(doc2vec)(2)_output/Adult Products_UI.feather")
print(df1[df1['UserID'] == 2815])
print(len(df1))
value_counts = df1["UserID"].value_counts()
print("고유값",len(value_counts))

df1 = pd.read_feather("../output/UR_output/Adult Products_UR.feather")
print(df1[df1['UserID'] == 2815])
print(len(df1))
value_counts = df1["UserID"].value_counts()
print("고유값",len(value_counts))



----------------------------------------------------------------------------------------------------
   UserID                                             Vector
0    2815  [0.22518493, 0.21682043, -0.08415891, 0.342974...
49
고유값 49
----------------------------------------------------------------------------------------------------
   UserID                                             Vector
1    2815  [-0.2635284662246704, -0.027993008494377136, -...
49
고유값 49
   UserID                                   Vector
0    2815  [-1.74607515335083, 1.7030137777328491]
49
고유값 49


### 폴더 순회 -> 필더명 변경

In [ ]:
import os
import pandas as pd

# A 폴더 경로
folder_path = '../output/URE(100dim)_output'

skipped_files=0
# A 폴더 내의 모든 파일을 순회
for filename in os.listdir(folder_path):
    if filename.endswith('.feather'):  # feather 파일만 처리
        file_path = os.path.join(folder_path, filename)
        try:
            df = pd.read_feather(file_path)

            if 'embedding' in df.columns:
                df.rename(columns={'embedding': 'Vector'}, inplace=True)
            
                # 변경된 데이터를 다시 feather 파일로 저장
                df.to_feather(file_path)
                # print(f"Updated {filename}")
        except Exception as e:
            # 읽을 수 없는 파일은 건너뛰기
            skipped_files += 1
            print(f"Skipping {filename} due to error: {e}")

# 건너뛴 파일 수 출력
print(f"\nTotal skipped files: {skipped_files}")


Total skipped files: 0


### vector열 형식 통일 (id : int, vector : 리스트)

In [3]:
import os
import pandas as pd

folder_path = '../output/UI(doc2vec)(2)_output'
feather_files = [f for f in os.listdir(folder_path) if f.endswith('.feather')]

def clean_userid(x):
    if isinstance(x, (float, int)):
        return int(x)
    elif isinstance(x, str):
        if '::' in x:
            x = x.split('::')[0]
        try:
            return int(float(x))
        except ValueError:
            return None
    else:
        return None

def convert_to_list(x):
    if isinstance(x, list):
        return x
    elif isinstance(x, str):
        try:
            return [float(val) for val in x.strip().split()]  # 공백 기준 split 후 float 변환
        except ValueError:
            return []
    elif hasattr(x, 'tolist'):
        return x.tolist()
    else:
        try:
            return list(x)
        except Exception:
            return []

for file in feather_files:
    file_path = os.path.join(folder_path, file)
    df = pd.read_feather(file_path)

    if 'UserID' in df.columns:
        df['UserID'] = df['UserID'].apply(clean_userid)

    vector_cols = [col for col in df.columns if 'Vector' in col]
    for col in vector_cols:
        df[col] = df[col].apply(convert_to_list)

    df.to_feather(file_path)

print("모든 Feather 파일에 대해 UserID 정리 및 Vector 문자열 → 리스트 변환 완료.")


모든 Feather 파일에 대해 UserID 정리 및 Vector 문자열 → 리스트 변환 완료.


### 파일 형식 통일

In [ ]:
import os

# 디렉토리 경로
folder_path = '../output/URE(BERT)_output'

# 폴더 내 모든 파일에 대해 반복
for filename in os.listdir(folder_path):
    if filename.endswith('.feather') and not filename.endswith('_URE.feather'):
        old_path = os.path.join(folder_path, filename)
        name_only = filename[:-8]  # .feather 제거
        new_filename = f"{name_only}_URE.feather"
        new_path = os.path.join(folder_path, new_filename)
        os.rename(old_path, new_path)
        print(f"Renamed: {filename} -> {new_filename}")
